In [ ]:
import os
import math
import json
import random
import gc
from pathlib import Path
from dataclasses import dataclass
from typing import Dict, List, Optional, Tuple, Any
import numpy as np
import pandas as pd
import scipy.io as sio
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from sklearn.model_selection import train_test_split
from sklearn.metrics import (cohen_kappa_score,accuracy_score)
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
def to_jsonable(obj):
    if isinstance(obj, Path):
        return str(obj)
    if isinstance(obj, np.generic):
        return obj.item()
    if isinstance(obj, dict):
        return {k: to_jsonable(v) for k, v in obj.items()}
    if isinstance(obj, (list, tuple)):
        return [to_jsonable(v) for v in obj]
    return obj
# ============================================================
# Hyperparameters
# ============================================================
USE_TORCH_COMPILE = True
COMPILE_MODE = "reduce-overhead"
COMPILE_FULLGRAPH = False
COMPILE_ENCODER_ONLY = True
SEED_LIST = [44] #[44,122,344]
HSID = "UH"
RESULT_ROOT = (Path.cwd() / f"{HSID}_Mamba_World_Results").resolve()
DATA_ROOT = (Path.cwd() / ".." / "HSI").resolve()
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if DEVICE.type == "cuda":
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
    torch.set_float32_matmul_precision("high")
def get_default_amp_dtype():
    if DEVICE.type != "cuda":
        return torch.float32
    major, _minor = torch.cuda.get_device_capability()
    return torch.bfloat16 if major >= 8 else torch.float16
AMP_DTYPE = get_default_amp_dtype() if DEVICE.type == "cuda" else torch.float32
def auto_batch_size(default: int = 32) -> int:
    if DEVICE.type != "cuda":
        return 16
    total_gb = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
    if total_gb >= 80:
        return 64
    if total_gb >= 40:
        return 48
    if total_gb >= 24:
        return 32
    if total_gb >= 16:
        return 24
    if total_gb >= 10:
        return 16
    return default
BATCH_SIZE = auto_batch_size()
EPOCHS = 100
PATCH_SIZE = 15
TRAIN_RATIO = 0.30
VAL_RATIO = 0.20
PCA_COMPONENTS = None
BASE_LR = 3e-4
WEIGHT_DECAY = 1e-4
LABEL_SMOOTHING = 0.00
EARLY_STOP_PATIENCE = 25
GRAD_CLIP_NORM = 1.0
USE_AMP = True
ACCUM_STEPS = 2  # micro-batch accumulation to reduce GPU memory pressure
NUM_WORKERS = min(8, max(2, (os.cpu_count() or 4) // 2))
PIN_MEMORY = DEVICE.type == "cuda"
BOOTSTRAP_ROUNDS = 200
BOUNDARY_DILATION = 2
# figure / result saving
SAVE_SCENE_MAPS = False
SAVE_PREDICTIONS_CSV = Flase
SAVE_METRICS_CSV = True
SAVE_CURVES = False
SAVE_CONFUSION_PLOTS = False
SAVE_TSNE = False
SAVE_BRANCH_ANALYSIS = False
SAVE_BRANCH_MAPS = False
SAVE_CLASS_ACC_CSV = False
SAVE_PAPER_TABLE = False
SAVE_FIGURE_SUFFIX = ".png"
# experiment controls
RUN_FULL_MODEL = True
RUN_ABLATIONS = False
RUN_PARAM_GRID = False
RUN_BRANCH_ANALYSIS_MODE = False
# ablation toggles used for the model
DEFAULT_BRANCH_FLAGS = dict(use_material=True, use_scene=True, use_phys=True)
# model-specific
MODEL_DIM = 64
MAMBA_LAYERS = 2
MAMBA_D_STATE = 8
MAMBA_D_CONV = 4
MAMBA_EXPAND = 1
STEM_KERNEL_DEPTH = 7
STEM_KERNEL_HW = 3
STEM_STRIDE_DEPTH = 2
STEM_STRIDE_HW = 2
BRANCH_DIM = 64
FACTOR_DIM = 16
DROPOUT = 0.10
ALPHA_MAT = 0.20
ALPHA_SCENE = 0.20
ALPHA_PHYS = 0.10
ALPHA_CONS = 0.05
ALPHA_SUPCON = 0.03
ALPHA_PROTO = 0.003
ALPHA_PHYS_REG = 0.02
CONS_TEMPERATURE = 1.0
SUPCON_TEMPERATURE = 0.1
PROTOTYPE_TEMPERATURE = 0.07
## parameter grid (set to None if you do not want a full sweep)
PATCH_SIZE_GRID = [9, 15, 25]
TRAIN_RATIO_GRID = [0.05, 0.10, 0.15]
RESULT_ROOT.mkdir(parents=True, exist_ok=True)
print("Torch:", torch.__version__)
print("Device:", DEVICE)
print("Data root:", DATA_ROOT)
print("Result root:", RESULT_ROOT)
print("AMP dtype:", AMP_DTYPE)
print("Auto batch size:", BATCH_SIZE)
print("Workers:", NUM_WORKERS)
def set_seed(seed_value: int = 345):
    random.seed(seed_value)
    np.random.seed(seed_value)
    torch.manual_seed(seed_value)
    torch.cuda.manual_seed_all(seed_value)
    torch.backends.cudnn.deterministic = False
    torch.backends.cudnn.benchmark = True
set_seed(SEED_LIST[0])
# ============================================================
# Data loading and preprocessing
# ============================================================
def LoadHSIData(method, data_root=DATA_ROOT):
    data_path = Path(data_root)
    if method == "UH":
        HSI = sio.loadmat(data_path / "HU.mat")["HSI"]
        GT = sio.loadmat(data_path / "HU_gt.mat")["gt"]
        target_names = ["Healthy grass","Stressed grass","Synthetic grass","Trees","Soil","Water","Residential","Commercial",
                        "Road","Highway","Railway","Parking Lot 1","Parking Lot 2","Tennis Court","Running Track"]
    else:
        raise ValueError(f"Unsupported dataset: {method}")
    return HSI.astype(np.float32), GT.astype(np.int32), len(target_names), target_names
## Normalization
def NormalizeHSI_MCMTN(HSI, eps=1e-6, clip_sigma=5.0):
    HSI = HSI.astype(np.float32)
    band_mean = HSI.mean(axis=(0, 1), keepdims=True)
    band_std = HSI.std(axis=(0, 1), keepdims=True)
    HSI = (HSI - band_mean) / (band_std + eps)
    HSI = np.clip(HSI, -clip_sigma, clip_sigma)
    return HSI.astype(np.float32)
## PCA function (if n_components is None, returns original HSI and None for PCA object)
def apply_pca_if_needed(hsi, n_components=None, seed=SEED_LIST[0]):
    if n_components is None:
        return hsi.astype(np.float32), None
    h, w, b = hsi.shape
    flat = hsi.reshape(-1, b)
    pca = PCA(n_components=n_components, random_state=seed)
    reduced = pca.fit_transform(flat).reshape(h, w, n_components).astype(np.float32)
    return reduced, pca
## Mirror padding
def mirror_pad_hsi(hsi, pad):
    return np.pad(hsi, ((pad, pad), (pad, pad), (0, 0)), mode="reflect")
## Function to extract labeled pixel indices and their corresponding labels from the GT map
def make_labeled_indices(gt):
    rows, cols = np.where(gt > 0)
    labels = gt[rows, cols].astype(np.int64) - 1
    return rows.astype(np.int64), cols.astype(np.int64), labels
## Stratified split of labeled indices into train, val, and test sets
def stratified_split_indices(rows, cols, labels, train_ratio=0.1, val_ratio=0.1, seed=SEED_LIST[0]):
    idx = np.arange(len(labels))
    train_idx, temp_idx = train_test_split(idx, train_size=train_ratio, random_state=seed, stratify=labels,)
    temp_labels = labels[temp_idx]
    val_size_in_temp = val_ratio / max(1e-8, 1.0 - train_ratio)
    val_idx, test_idx = train_test_split(temp_idx, train_size=val_size_in_temp, random_state=seed, stratify=temp_labels,)
    return train_idx, val_idx, test_idx
## Custom Dataset class for extracting patches centered around labeled pixels
class HSIPatchDataset(Dataset):
    def __init__(self, hsi, gt, indices, patch_size=13):
        self.hsi = hsi.astype(np.float32)
        self.gt = gt.astype(np.int32)
        self.indices = np.asarray(indices, dtype=np.int64)
        self.patch_size = patch_size
        self.pad = patch_size // 2
        rows, cols, labels = make_labeled_indices(gt)
        self.rows = rows[self.indices]
        self.cols = cols[self.indices]
        self.labels = labels[self.indices]
        self.hsi_pad = mirror_pad_hsi(self.hsi, self.pad)
    def __len__(self):
        return len(self.indices)
    def _get_patch(self, i):
        r = int(self.rows[i])
        c = int(self.cols[i])
        y = int(self.labels[i])
        rp, cp = r + self.pad, c + self.pad
        patch = self.hsi_pad[rp - self.pad: rp + self.pad + 1,cp - self.pad: cp + self.pad + 1,:]
        patch = np.transpose(patch, (2, 0, 1)).astype(np.float32)
        return (torch.from_numpy(patch),torch.tensor(y, dtype=torch.long),torch.tensor([r, c], dtype=torch.long),)
    def __getitem__(self, i):
        return self._get_patch(i)
## Dataset class that returns patches for all pixels in the scene
class HSIFullSceneDataset(Dataset):
    def __init__(self, hsi, gt, patch_size=13):
        self.hsi = hsi.astype(np.float32)
        self.gt = gt.astype(np.int32)
        self.patch_size = patch_size
        self.pad = patch_size // 2
        self.h, self.w, self.b = self.hsi.shape
        self.hsi_pad = mirror_pad_hsi(self.hsi, self.pad)
        self.rows = np.arange(self.h)
        self.cols = np.arange(self.w)
        self.grid = [(r, c) for r in range(self.h) for c in range(self.w)]
    def __len__(self):
        return self.h * self.w
    def __getitem__(self, idx):
        r, c = self.grid[idx]
        rp, cp = r + self.pad, c + self.pad
        patch = self.hsi_pad[rp - self.pad: rp + self.pad + 1, cp - self.pad: cp + self.pad + 1, :,]
        patch = np.transpose(patch, (2, 0, 1)).astype(np.float32)
        y = int(self.gt[r, c] - 1) if self.gt[r, c] > 0 else -1
        return torch.from_numpy(patch), torch.tensor(y, dtype=torch.long), torch.tensor([r, c], dtype=torch.long)
# ============================================================
# Saving helpers
# ============================================================
def ensure_dir(path: Path):
    path.mkdir(parents=True, exist_ok=True)
## Helper function to format mean and std
def format_mean_std(mean: float, std: float, digits: int = 4) -> str:
    return f"{mean:.{digits}f} ± {std:.{digits}f}"
def compute_light_metrics(y_true: np.ndarray, y_pred: np.ndarray) -> Dict[str, float]:
    num_classes = int(np.max(y_true)) + 1 if len(y_true) else 0
    oa = accuracy_score(y_true, y_pred) if len(y_true) else np.nan
    aa = np.nanmean([
        accuracy_score(y_true[y_true == c], y_pred[y_true == c]) if np.any(y_true == c) else np.nan
        for c in range(num_classes)
    ]) if num_classes > 0 else np.nan
    kappa = cohen_kappa_score(y_true, y_pred) if len(y_true) else np.nan
    return {"oa": float(oa),"aa": float(aa),"kappa": float(kappa),}
## Function to save predictions and related info in a CSV file
def save_prediction_csv(out_dir: Path, coords: np.ndarray, y_true: np.ndarray, y_pred: np.ndarray, y_prob: np.ndarray):
    df = pd.DataFrame({"row": coords[:, 0].astype(int),"col": coords[:, 1].astype(int),"y_true": y_true.astype(int),
                       "y_pred": y_pred.astype(int),"correct": (y_true == y_pred).astype(int),"confidence": y_prob.max(axis=1),})
    df.to_csv(out_dir / "predictions.csv", index=False)
## Function to save a color-coded label map of the scene
import matplotlib.colors as mcolors
def save_label_map_png(label_map: np.ndarray,title: str,save_path: Path,num_classes: int,black_background: bool = False,):
    fig, ax = plt.subplots(figsize=(8, 6))
    cmap = plt.get_cmap("nipy_spectral", max(num_classes + 1, 2)).copy()
    if black_background:
        colors = cmap(np.arange(cmap.N))
        colors[0] = [0.0, 0.0, 0.0, 1.0]  # class 0 -> black
        cmap = mcolors.ListedColormap(colors)
    im = ax.imshow(label_map,cmap=cmap,interpolation="nearest",vmin=0,vmax=max(num_classes, 1),)
    ax.set_title(title)
    ax.axis("off")
    fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    plt.tight_layout()
    fig.savefig(save_path, dpi=300, bbox_inches="tight")
    plt.close(fig)
# ============================================================
# Mamba encoder with state branches
# ============================================================
class Tokenizer3D(nn.Module):
    def __init__(self,in_bands: int,embed_dim: int,patch_size: int,d_kernel=7,h_kernel=3,w_kernel=3,d_stride=2,h_stride=2,w_stride=2):
        super().__init__()
        self.in_bands = in_bands
        self.embed_dim = embed_dim
        self.patch_size = patch_size
        self.d_kernel = min(d_kernel, in_bands)
        self.d_stride = max(1, min(d_stride, self.d_kernel))
        self.h_kernel = h_kernel
        self.w_kernel = w_kernel
        self.h_stride = h_stride
        self.w_stride = w_stride
        pad_d = self.d_kernel // 2
        pad_h = self.h_kernel // 2
        pad_w = self.w_kernel // 2
        mid = max(32, embed_dim // 2)
        self.stem = nn.Sequential(nn.Conv3d(1, mid, kernel_size=(self.d_kernel, self.h_kernel, self.w_kernel),
                                            stride=(self.d_stride, self.h_stride, self.w_stride),padding=(pad_d, pad_h, pad_w), bias=False),
                                  nn.BatchNorm3d(mid),nn.GELU(),nn.Conv3d(mid, embed_dim, kernel_size=1, bias=False),nn.BatchNorm3d(embed_dim),nn.GELU(),)
    def forward(self, x: torch.Tensor) -> Tuple[torch.Tensor, Tuple[int, int, int]]:
        x = x.unsqueeze(1)
        feat = self.stem(x)
        b, c, d, h, w = feat.shape
        tokens = feat.flatten(2).transpose(1, 2).contiguous()
        return tokens, (d, h, w)
class FastMamba(nn.Module):
    def __init__(self,d_model: int,d_state: int = 16,d_conv: int = 4,expand: int = 2,dt_rank: Optional[int] = None,dropout: float = 0.0,):
        super().__init__()
        self.d_model = int(d_model)
        self.d_state = int(d_state)
        self.d_conv = int(d_conv)
        self.expand = int(expand)
        self.d_inner = self.d_model * self.expand
        self.dt_rank = int(dt_rank if dt_rank is not None else max(1, math.ceil(self.d_model / 16)))
        self.in_proj = nn.Linear(self.d_model, 2 * self.d_inner, bias=True)
        self.conv1d = nn.Conv1d(self.d_inner,self.d_inner,kernel_size=self.d_conv,groups=self.d_inner,bias=True,padding=0,)
        self.x_proj = nn.Linear(self.d_inner, self.dt_rank + 2 * self.d_state, bias=False)
        self.dt_proj = nn.Linear(self.dt_rank, self.d_inner, bias=True)
        self.out_proj = nn.Linear(self.d_inner, self.d_model, bias=True)
        self.dropout = nn.Dropout(dropout)
        A_init = torch.arange(1, self.d_state + 1, dtype=torch.float32)
        A_init = torch.log(A_init).unsqueeze(0).repeat(self.d_inner, 1)
        self.A_log = nn.Parameter(A_init)
        self.D = nn.Parameter(torch.ones(self.d_inner))
        self.dt_bias = nn.Parameter(torch.zeros(self.d_inner))
        self.reset_parameters()
    def reset_parameters(self):
        nn.init.xavier_uniform_(self.in_proj.weight)
        nn.init.zeros_(self.in_proj.bias)
        nn.init.kaiming_uniform_(self.conv1d.weight, a=math.sqrt(5))
        nn.init.zeros_(self.conv1d.bias)
        nn.init.xavier_uniform_(self.x_proj.weight)
        nn.init.xavier_uniform_(self.dt_proj.weight)
        nn.init.zeros_(self.dt_proj.bias)
        nn.init.xavier_uniform_(self.out_proj.weight)
        nn.init.zeros_(self.out_proj.bias)
    def _causal_depthwise_conv(self, x: torch.Tensor) -> torch.Tensor:
        if self.d_conv > 1:
            x = F.pad(x, (self.d_conv - 1, 0))
        return self.conv1d(x)
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        if x.dim() != 3:
            raise ValueError(f"FastMamba expects [B, L, C], got {tuple(x.shape)}")
        B, L, _ = x.shape
        xz = self.in_proj(x)
        x_branch, z_branch = xz.chunk(2, dim=-1)
        x_branch = x_branch.transpose(1, 2).contiguous()
        x_branch = self._causal_depthwise_conv(x_branch)
        x_branch = F.silu(x_branch).transpose(1, 2).contiguous()
        params = self.x_proj(x_branch)
        dt_raw, b_raw, c_raw = torch.split(params, [self.dt_rank, self.d_state, self.d_state], dim=-1)
        dt = F.softplus(self.dt_proj(dt_raw) + self.dt_bias.view(1, 1, -1)) + 1e-4
        b_token = torch.tanh(b_raw)
        c_token = torch.tanh(c_raw)
        A = -torch.exp(self.A_log)
        log_a = dt.unsqueeze(-1) * A.unsqueeze(0)
        u = dt.unsqueeze(-1) * x_branch.unsqueeze(-1) * b_token.unsqueeze(2)
        log_prefix = torch.cumsum(log_a, dim=1)
        shift = log_prefix.max(dim=1, keepdim=True).values
        prefix = torch.exp(log_prefix - shift)
        eps = 1e-6
        state = prefix * torch.cumsum(u / (prefix + eps), dim=1)
        y = (state * c_token.unsqueeze(2)).sum(dim=-1) + self.D.view(1, 1, -1) * x_branch
        y = y * torch.sigmoid(z_branch)
        y = self.dropout(y)
        return self.out_proj(y)
class FastMambaBlock(nn.Module):
    def __init__(self, d_model: int, d_state: int = 16, d_conv: int = 4, expand: int = 2, drop: float = 0.0):
        super().__init__()
        self.norm = nn.LayerNorm(d_model)
        self.mamba = FastMamba(d_model=d_model, d_state=d_state, d_conv=d_conv, expand=expand, dropout=drop)
        self.drop = nn.Dropout(drop)
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return x + self.drop(self.mamba(self.norm(x)))
class FastMambaStack(nn.Module):
    def __init__(self, d_model: int, depth: int, d_state: int, d_conv: int, expand: int, drop: float):
        super().__init__()
        self.blocks = nn.ModuleList([FastMambaBlock(d_model=d_model, d_state=d_state, d_conv=d_conv, expand=expand, drop=drop) for _ in range(depth)])
        self.final_norm = nn.LayerNorm(d_model)
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        for blk in self.blocks:
            x = blk(x)
        return self.final_norm(x)
## Atten. Pooling
class AttentionPool(nn.Module):
    def __init__(self, dim: int, hidden: Optional[int] = None):
        super().__init__()
        hidden = hidden or max(dim // 2, 32)
        self.score = nn.Sequential(nn.Linear(dim, hidden),nn.GELU(),nn.Linear(hidden, 1),)
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        a = self.score(x)
        a = torch.softmax(a, dim=1)
        return (a * x).sum(dim=1)
## Physical factor head that predicts physical factors and class logits from the model's representation
class PhysicalFactorHead(nn.Module):
    def __init__(self, in_dim: int, factor_dim: int, num_classes: int, dropout: float = 0.1):
        super().__init__()
        self.net = nn.Sequential(nn.LayerNorm(in_dim), nn.Linear(in_dim, in_dim), 
                                 nn.GELU(), nn.Dropout(dropout), nn.Linear(in_dim, factor_dim),)
        self.classifier = nn.Linear(factor_dim, num_classes)
    def forward(self, x: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
        z = self.net(x)
        logits = self.classifier(z)
        return z, logits
## Function to compute cosine similarity logits between input features and class prototypes
def cosine_logits(x: torch.Tensor, prototypes: torch.Tensor, temperature: float = 0.07) -> torch.Tensor:
    x = F.normalize(x, dim=-1)
    p = F.normalize(prototypes, dim=-1)
    return (x @ p.T) / temperature
## The main model class that combines the tokenizer, Mamba stack, attention pooling, and classification heads
class HSIWorldMambaNet(nn.Module):
    def __init__(self, bands: int, num_classes: int, patch_size: int,
                 model_dim: int = 128, depth: int = 4,
                 d_state: int = 16, d_conv: int = 4, expand: int = 2,
                 factor_dim: int = 32, drop: float = 0.1,
                 use_material: bool = True, use_scene: bool = True, use_phys: bool = True):
        super().__init__()
        self.num_classes = num_classes
        self.model_dim = model_dim
        self.factor_dim = factor_dim
        self.use_material = use_material
        self.use_scene = use_scene
        self.use_phys = use_phys
        self.tokenizer = Tokenizer3D(in_bands=bands,embed_dim=model_dim,patch_size=patch_size,d_kernel=STEM_KERNEL_DEPTH,
                                     h_kernel=STEM_KERNEL_HW,w_kernel=STEM_KERNEL_HW,d_stride=STEM_STRIDE_DEPTH,
                                     h_stride=STEM_STRIDE_HW,w_stride=STEM_STRIDE_HW,)
        with torch.no_grad():
            dummy = torch.zeros(1, bands, patch_size, patch_size)
            _, grid = self.tokenizer(dummy)
        self.grid_shape = grid
        d, h, w = grid
        n_tokens = d * h * w
        self.cls_token = nn.Parameter(torch.zeros(1, 1, model_dim))
        self.pos_embed = nn.Parameter(torch.zeros(1, n_tokens + 1, model_dim))
        self.pos_drop = nn.Dropout(drop)
        self.encoder = FastMambaStack(d_model=model_dim,depth=depth,d_state=d_state,d_conv=d_conv,expand=expand,drop=drop,)
        self.spectral_pool = AttentionPool(model_dim)
        self.spatial_pool = AttentionPool(model_dim)
        self.global_pool = AttentionPool(model_dim)
        self.cls_head = nn.Sequential(nn.LayerNorm(model_dim), nn.Linear(model_dim, model_dim), 
                                      nn.GELU(), nn.Dropout(drop), nn.Linear(model_dim, num_classes),)
        self.material_proj = nn.Sequential(nn.LayerNorm(model_dim), 
                                           nn.Linear(model_dim, BRANCH_DIM), nn.GELU(), nn.Dropout(drop),)
        self.scene_proj = nn.Sequential(nn.LayerNorm(model_dim), 
                                        nn.Linear(model_dim, BRANCH_DIM), nn.GELU(), nn.Dropout(drop),)
        self.material_prototypes = nn.Parameter(torch.randn(num_classes, BRANCH_DIM) * 0.02)
        self.scene_prototypes = nn.Parameter(torch.randn(num_classes, BRANCH_DIM) * 0.02)
        self.prototype_temp = PROTOTYPE_TEMPERATURE
        self.physical_head = PhysicalFactorHead(model_dim, factor_dim, num_classes, dropout=drop)
        self.branch_fusion_logits = nn.Parameter(torch.zeros(4))
        self.dropout = nn.Dropout(drop)
        self.reset_parameters()
    def reset_parameters(self):
        nn.init.trunc_normal_(self.cls_token, std=0.02)
        nn.init.trunc_normal_(self.pos_embed, std=0.02)
        nn.init.trunc_normal_(self.material_prototypes, std=0.02)
        nn.init.trunc_normal_(self.scene_prototypes, std=0.02)
    def _reshape_grid(self, tok: torch.Tensor) -> torch.Tensor:
        b, n, c = tok.shape
        d, h, w = self.grid_shape
        assert n == d * h * w, (n, d, h, w)
        return tok.transpose(1, 2).contiguous().view(b, c, d, h, w)
    def forward(self, x: torch.Tensor) -> Dict[str, torch.Tensor]:
        tokens, grid = self.tokenizer(x)
        if grid != self.grid_shape:
            raise RuntimeError(f"Token grid mismatch: got {grid}, expected {self.grid_shape}")
        b, n, c = tokens.shape
        cls = self.cls_token.expand(b, -1, -1)
        seq = torch.cat([cls, tokens], dim=1)
        seq = self.pos_drop(seq + self.pos_embed)
        seq = self.encoder(seq)
        cls_tok = seq[:, 0]
        tok = seq[:, 1:]
        feat = self._reshape_grid(tok)
        z_global = self.global_pool(tok)
        logits_main = self.cls_head(self.dropout(cls_tok))
        logits_material = None
        z_material = None
        if self.use_material:
            spectral_tokens = feat.mean(dim=(3, 4)).transpose(1, 2).contiguous()
            z_material_base = self.spectral_pool(spectral_tokens)
            z_material = self.material_proj(z_material_base)
            logits_material = cosine_logits(z_material, self.material_prototypes, temperature=self.prototype_temp)
        logits_scene = None
        z_scene = None
        if self.use_scene:
            spatial_tokens = feat.mean(dim=2).flatten(2).transpose(1, 2).contiguous()
            z_scene_base = self.spatial_pool(spatial_tokens)
            z_scene = self.scene_proj(z_scene_base)
            logits_scene = cosine_logits(z_scene, self.scene_prototypes, temperature=self.prototype_temp)
        logits_phys = None
        z_phys = None
        if self.use_phys:
            z_phys, logits_phys = self.physical_head(z_global)
        branch_logits = [logits_main]
        if logits_material is not None:
            branch_logits.append(logits_material)
        if logits_scene is not None:
            branch_logits.append(logits_scene)
        if logits_phys is not None:
            branch_logits.append(logits_phys)
        if len(branch_logits) == 1:
            logits_fused = logits_main
        else:
            weights = torch.softmax(self.branch_fusion_logits[:len(branch_logits)], dim=0)
            logits_fused = sum(w * l for w, l in zip(weights, branch_logits))
        relation_matrix = F.normalize(self.material_prototypes, dim=-1) @ F.normalize(self.material_prototypes, dim=-1).T
        return {'logits': logits_fused,'logits_main': logits_main,'logits_material': logits_material,'logits_scene': logits_scene,
                'logits_phys': logits_phys,'z_global': z_global,'z_material': z_material,'z_scene': z_scene,'z_phys': z_phys,
                'cls_token': cls_tok,'material_prototypes': self.material_prototypes,'scene_prototypes': self.scene_prototypes,
                'relation_matrix': relation_matrix,'feat_grid': feat,}
# ============================================================
# Supervised auxiliary losses
# ============================================================
class SupervisedContrastiveLoss(nn.Module):
    def __init__(self, temperature=0.1):
        super().__init__()
        self.temperature = temperature
    def forward(self, features: torch.Tensor, labels: torch.Tensor) -> torch.Tensor:
        if features.dim() == 2:
            features = features.unsqueeze(1)
        bsz, n_views, dim = features.shape
        feats = F.normalize(features.reshape(bsz * n_views, dim), dim=-1)
        labels = labels.view(-1, 1)
        mask = torch.eq(labels, labels.T).float().to(features.device)
        contrast_count = n_views
        contrast_feature = feats
        anchor_feature = contrast_feature
        anchor_count = contrast_count
        logits = torch.div(anchor_feature @ contrast_feature.T, self.temperature)
        logits_max, _ = torch.max(logits, dim=1, keepdim=True)
        logits = logits - logits_max.detach()
        mask = mask.repeat(anchor_count, contrast_count)
        logits_mask = torch.ones_like(mask)
        logits_mask.fill_diagonal_(0)
        mask = mask * logits_mask
        exp_logits = torch.exp(logits) * logits_mask
        log_prob = logits - torch.log(exp_logits.sum(dim=1, keepdim=True) + 1e-12)
        mean_log_prob_pos = (mask * log_prob).sum(dim=1) / (mask.sum(dim=1) + 1e-12)
        loss = -mean_log_prob_pos.view(anchor_count, bsz).mean()
        return loss
## Physical factor loss that encourages low variance, low covariance, and within-class compactness
def off_diagonal(x: torch.Tensor) -> torch.Tensor:
    n, m = x.shape
    assert n == m
    return x.flatten()[:-1].view(n - 1, n + 1)[:, 1:].flatten()
## Covariance matrix of a given tensor
def covariance_matrix(z: torch.Tensor) -> torch.Tensor:
    z = z - z.mean(dim=0, keepdim=True)
    n = z.shape[0]
    return (z.T @ z) / max(n - 1, 1)
## Variance margin loss
def variance_margin_loss(z: torch.Tensor, margin: float = 1.0) -> torch.Tensor:
    std = torch.sqrt(z.var(dim=0, unbiased=False) + 1e-4)
    return torch.mean(F.relu(margin - std))
## Within-class compactness loss
def within_class_compactness(z: torch.Tensor, y: torch.Tensor) -> torch.Tensor:
    uniq = torch.unique(y)
    losses = []
    for c in uniq:
        idx = (y == c)
        if idx.sum() < 2:
            continue
        zc = z[idx]
        mu = zc.mean(dim=0, keepdim=True)
        losses.append(((zc - mu) ** 2).mean())
    if len(losses) == 0:
        return z.new_tensor(0.0)
    return torch.stack(losses).mean()
## Combined physical factor loss that includes variance margin, covariance, and within-class compactness
class PhysicalFactorLoss(nn.Module):
    def __init__(self, var_margin=1.0, cov_weight=1.0, compact_weight=1.0):
        super().__init__()
        self.var_margin = var_margin
        self.cov_weight = cov_weight
        self.compact_weight = compact_weight
    def forward(self, z_phys: torch.Tensor, labels: torch.Tensor) -> torch.Tensor:
        var_loss = variance_margin_loss(z_phys, margin=self.var_margin)
        cov = covariance_matrix(z_phys)
        cov_loss = (off_diagonal(cov).pow(2)).mean()
        compact = within_class_compactness(z_phys, labels)
        return var_loss + self.cov_weight * cov_loss + self.compact_weight * compact
## Prototype diversity loss that encourages class prototypes to be diverse and not collapse
class PrototypeDiversityLoss(nn.Module):
    def forward(self, protos: torch.Tensor) -> torch.Tensor:
        p = F.normalize(protos, dim=-1)
        sim = p @ p.T
        eye = torch.eye(sim.shape[0], device=sim.device, dtype=sim.dtype)
        return ((sim - eye) ** 2).mean()
## Symmetric KL divergence loss between two sets of logits, used for consistency regularization
def symmetric_kl(p_logits: torch.Tensor, q_logits: torch.Tensor, temperature: float = 1.0) -> torch.Tensor:
    p = F.log_softmax(p_logits / temperature, dim=-1)
    q = F.log_softmax(q_logits / temperature, dim=-1)
    p_prob = p.exp()
    q_prob = q.exp()
    kl_pq = F.kl_div(p, q_prob, reduction='batchmean')
    kl_qp = F.kl_div(q, p_prob, reduction='batchmean')
    return 0.5 * (kl_pq + kl_qp)
# ============================================================
# Train / eval loops
# ============================================================
def branch_loss_or_zero(criterion, logits, y):
    if logits is None:
        return y.new_tensor(0.0, dtype=torch.float32)
    return criterion(logits, y)
def run_batch_single(model, batch, criterion, supcon, phys_criterion, proto_div, cfg):
    x, y, _ = batch
    x = x.to(DEVICE, non_blocking=True)
    y = y.to(DEVICE, non_blocking=True)
    out = model(x)
    ce_main = criterion(out["logits_main"], y)
    ce_mat = branch_loss_or_zero(criterion, out["logits_material"], y)
    ce_scene = branch_loss_or_zero(criterion, out["logits_scene"], y)
    ce_phys = branch_loss_or_zero(criterion, out["logits_phys"], y)
    ce_fused = criterion(out["logits"], y)
    cons = y.new_tensor(0.0, dtype=torch.float32)
    supcon_loss = y.new_tensor(0.0, dtype=torch.float32)
    if cfg.get("use_supcon", True):
        feats_list = [out["cls_token"]]
        if out["z_material"] is not None:
            feats_list.append(out["z_material"])
        if out["z_scene"] is not None:
            feats_list.append(out["z_scene"])
        feats = torch.stack(feats_list, dim=1)
        supcon_loss = supcon(feats, y)
    phys_loss = y.new_tensor(0.0, dtype=torch.float32)
    if out["z_phys"] is not None:
        phys_loss = phys_criterion(out["z_phys"], y)
    proto_loss = y.new_tensor(0.0, dtype=torch.float32)
    if out["material_prototypes"] is not None:
        proto_loss = 0.5 * (proto_div(out["material_prototypes"]) + proto_div(out["scene_prototypes"]))
    total = (ce_fused + 0.5 * ce_main + ALPHA_MAT * ce_mat + ALPHA_SCENE * ce_scene + ALPHA_PHYS * ce_phys + 
             ALPHA_CONS * cons + ALPHA_SUPCON * supcon_loss + ALPHA_PROTO * proto_loss + ALPHA_PHYS_REG * phys_loss)
    return {"loss": total,"ce_fused": ce_fused.detach(),"ce_main": ce_main.detach(),"ce_mat": ce_mat.detach(),
            "ce_scene": ce_scene.detach(),"ce_phys": ce_phys.detach(),"cons": cons.detach(),"supcon": supcon_loss.detach(),
            "phys": phys_loss.detach(),"proto": proto_loss.detach(),}
## Evaluation function that runs the model on a data loader and collects predictions and probabilities
@torch.inference_mode()
def evaluate(model, loader):
    model.eval()
    ys, preds, probs, coords = [], [], [], []
    for batch in loader:
        x, y, rc = batch
        x = x.to(DEVICE, non_blocking=True)
        y = y.to(DEVICE, non_blocking=True)
        out = model(x)
        p = torch.softmax(out['logits'], dim=-1)
        pred = p.argmax(dim=-1)
        ys.append(y.cpu().numpy())
        preds.append(pred.cpu().numpy())
        probs.append(p.cpu().numpy())
        coords.append(rc.numpy())
    y_true = np.concatenate(ys)
    y_pred = np.concatenate(preds)
    y_prob = np.concatenate(probs)
    coords = np.concatenate(coords)
    return y_true, y_pred, y_prob, coords
## Brnch Accuracy
@torch.no_grad()
def evaluate_branch_accuracy(model, loader):
    model.eval()
    total=0
    stats={"Main":0,"Material":0,"Scene":0,"Physical":0,"Fused":0,}
    for x,y,_ in loader:
        x=x.to(DEVICE)
        y=y.to(DEVICE)
        out=model(x)
        total+=y.size(0)
        stats["Fused"] += (out["logits"].argmax(1)==y).sum().item()
        stats["Main"] += (out["logits_main"].argmax(1)==y).sum().item()
        if out["logits_material"] is not None:
            stats["Material"] += (out["logits_material"].argmax(1)==y).sum().item()
        if out["logits_scene"] is not None:
            stats["Scene"] += (out["logits_scene"].argmax(1)==y).sum().item()
        if out["logits_phys"] is not None:
            stats["Physical"] += (out["logits_phys"].argmax(1)==y).sum().item()
    for k in stats:
        stats[k]/=total
    return stats
## Evaluate metrics given true labels, predictions, probabilities, and class names
def evaluate_metrics(model, loader):
    y_true, y_pred, y_prob, coords = evaluate(model, loader)
    metrics = compute_light_metrics(y_true, y_pred)
    return metrics, y_true, y_pred, y_prob, coords
def get_class_weights(train_labels: np.ndarray, num_classes: int) -> torch.Tensor:
    counts = np.bincount(train_labels, minlength=num_classes).astype(np.float32)
    weights = counts.sum() / np.maximum(counts, 1.0)
    weights = weights / weights.mean()
    return torch.tensor(weights, dtype=torch.float32)
## Main training loop that trains the model, evaluates on validation set, and saves the best model based on validation OA
def train_model(model, train_loader, val_loader, test_loader, class_names, cfg, out_dir: Path):
    ensure_dir(out_dir)
    class_weights = get_class_weights(train_loader.dataset.labels, len(class_names)).to(DEVICE)
    criterion = nn.CrossEntropyLoss(weight=class_weights, label_smoothing=LABEL_SMOOTHING)
    supcon = SupervisedContrastiveLoss(temperature=SUPCON_TEMPERATURE)
    phys_criterion = PhysicalFactorLoss(var_margin=1.0, cov_weight=0.5, compact_weight=0.5)
    proto_div = PrototypeDiversityLoss()
    optimizer = torch.optim.AdamW(model.parameters(), lr=BASE_LR, weight_decay=WEIGHT_DECAY)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)
    scaler = torch.amp.GradScaler('cuda', enabled=(USE_AMP and DEVICE.type == 'cuda'))
    best_val = -1.0
    best_epoch = -1
    bad_epochs = 0
    history = []
    best_path = out_dir / 'best_model.pt'
    for epoch in range(1, EPOCHS + 1):
        model.train()
        running = {}
        n_seen = 0
        optimizer.zero_grad(set_to_none=True)
        for step, batch in enumerate(train_loader, start=1):
            with torch.autocast(device_type=DEVICE.type, dtype=AMP_DTYPE, enabled=(USE_AMP and DEVICE.type == 'cuda')):
                out = run_batch_single(model, batch, criterion, supcon, phys_criterion, proto_div, cfg)
                loss = out['loss'] / ACCUM_STEPS
            scaler.scale(loss).backward()
            bs = batch[0].shape[0]
            n_seen += bs
            for k, v in out.items():
                running[k] = running.get(k, 0.0) + float(v) * bs
            if step % ACCUM_STEPS == 0:
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP_NORM)
                scaler.step(optimizer)
                scaler.update()
                optimizer.zero_grad(set_to_none=True)
        # flush any remaining gradients
        if n_seen > 0 and (len(train_loader) % ACCUM_STEPS) != 0:
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP_NORM)
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad(set_to_none=True)
        scheduler.step()
        train_stats = {k: v / max(n_seen, 1) for k, v in running.items()}
        val_metrics, _, _, _, _ = evaluate_metrics(model, val_loader)
        history.append({'epoch': epoch,'train_loss': train_stats.get('loss', np.nan),'train_ce_fused': train_stats.get('ce_fused', np.nan),
                        'train_cons': train_stats.get('cons', np.nan),'train_phys': train_stats.get('phys', np.nan),'val_oa': val_metrics['oa'],
                        'val_aa': val_metrics['aa'],'val_kappa': val_metrics['kappa'],})
        print(f"Epoch {epoch:03d} | Loss {history[-1]['train_loss']:.4f} | "
            f"Val OA {val_metrics['oa']:.4f} | AA {val_metrics['aa']:.4f} | Kappa {val_metrics['kappa']:.4f}")
        if val_metrics['oa'] > best_val:
            best_val = val_metrics['oa']
            best_epoch = epoch
            bad_epochs = 0
            torch.save({'model_state_dict': model.state_dict(),'best_val_oa': best_val,'epoch': epoch,}, best_path)
        else:
            bad_epochs += 1
            if bad_epochs >= EARLY_STOP_PATIENCE:
                print(f"Early stopping at epoch {epoch}")
                break
    ckpt = torch.load(best_path, map_location=DEVICE)
    model.load_state_dict(ckpt['model_state_dict'])
    test_metrics, y_true, y_pred, y_prob, coords = evaluate_metrics(model, test_loader)
    print('Best epoch:', best_epoch)
    print('Test OA:', test_metrics['oa'])
    print('Test AA:', test_metrics['aa'])
    print('Test Kappa:', test_metrics['kappa'])
    return model, history, test_metrics, y_true, y_pred, y_prob, coords, best_epoch, best_val
# ============================================================
# Experiment helpers
# ============================================================
def build_experiment_list():
    experiments = []
    if RUN_FULL_MODEL:
        experiments.append({'name': 'full_model','patch_size': PATCH_SIZE,'train_ratio': TRAIN_RATIO,'val_ratio': VAL_RATIO,
                            'use_material': True,'use_scene': True,'use_phys': True,'use_consistency': True,'use_supcon': True,})
    return experiments
@torch.no_grad()
def predict_full_scene(model, hsi, gt, patch_size: int, batch_size: int = 128):
    model.eval()
    full_ds = HSIFullSceneDataset(hsi, gt, patch_size=patch_size)
    full_loader = DataLoader(full_ds, batch_size=batch_size, shuffle=False, num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY)
    preds = np.zeros((hsi.shape[0], hsi.shape[1]), dtype=np.int32)
    probs = np.zeros((hsi.shape[0], hsi.shape[1], model.num_classes), dtype=np.float32)
    coords = []
    y_pred = []
    for x, y, rc in full_loader:
        x = x.to(DEVICE, non_blocking=True)
        out = model(x)
        p = torch.softmax(out['logits'], dim=-1).cpu().numpy()
        pred = p.argmax(axis=1)
        rc = rc.numpy()
        for i in range(len(rc)):
            r, c = int(rc[i, 0]), int(rc[i, 1])
            preds[r, c] = int(pred[i] + 1)
            probs[r, c] = p[i]
        coords.append(rc)
        y_pred.append(pred)
    return preds, probs, full_ds
def save_prediction_maps(model,hsi,gt,exp_dir: Path,patch_size: int,class_names: List[str],):
    pred_map, _, _ = predict_full_scene(model,hsi,gt,patch_size=patch_size,batch_size=BATCH_SIZE,)
    ensure_dir(exp_dir)
    save_label_map_png(gt,"Ground Truth",exp_dir / f"gt_full{SAVE_FIGURE_SUFFIX}",len(class_names),)
    save_label_map_png(pred_map,"Prediction",exp_dir / f"pred_full{SAVE_FIGURE_SUFFIX}",len(class_names),)
def save_seed_run_artifacts(exp_dir: Path,seed: int,metrics,y_true,y_pred,y_prob,coords,):
    seed_dir = exp_dir / f"seed_{seed}"
    ensure_dir(seed_dir)
    if SAVE_METRICS_CSV:
        metrics_df = pd.DataFrame([{"seed": seed,"oa": metrics["oa"],"aa": metrics["aa"],"kappa": metrics["kappa"],}])
        metrics_df.to_csv(seed_dir / "metrics.csv", index=False)
    if SAVE_PREDICTIONS_CSV:
        save_prediction_csv(seed_dir, coords, y_true, y_pred, y_prob)
    return seed_dir
def summarize_seed_results(seed_records: List[Dict[str, Any]],exp_dir: Path,):
    summary_df = pd.DataFrame(seed_records)
    summary_df.to_csv(exp_dir / "seed_wise_results.csv",index=False,)
    return summary_df
# ============================================================
# Main experiment runner
# ============================================================
def maybe_compile_model(model: nn.Module) -> nn.Module:
    if not USE_TORCH_COMPILE or not hasattr(torch, "compile"):
        return model
    try:
        torch._dynamo.config.suppress_errors = True
        try:
            import torch._inductor.config as inductor_config
            inductor_config.triton.cudagraphs = False
        except Exception:
            pass
        if COMPILE_ENCODER_ONLY and hasattr(model, "encoder"):
            model.encoder = torch.compile(model.encoder,mode=COMPILE_MODE,fullgraph=False,dynamic=True,)
            return model
        return torch.compile(model,mode=COMPILE_MODE,fullgraph=False,dynamic=True,)
    except Exception as e:
        print(f"[compile disabled] Falling back to eager mode: {e}")
        return model
def make_loader(dataset, batch_size, shuffle=False, drop_last=False):
    kwargs = dict(dataset=dataset,batch_size=batch_size,shuffle=shuffle,num_workers=NUM_WORKERS,pin_memory=PIN_MEMORY,drop_last=drop_last,)
    if NUM_WORKERS > 0:
        kwargs["persistent_workers"] = True
        kwargs["prefetch_factor"] = 4
    return DataLoader(**kwargs)
def run_experiment(exp_cfg: Dict[str, Any], seeds: List[int], hsi, gt, class_names):
    exp_name = exp_cfg['name']
    exp_dir = RESULT_ROOT / exp_name
    ensure_dir(exp_dir)
    with open(exp_dir / 'config.json', 'w') as f:
        json.dump(exp_cfg, f, indent=2)
    seed_records = []
    best_seed_info = None
    best_seed_val_oa = -1.0
    best_seed_model = None
    best_seed_test_pack = None
    for seed in seeds:
        print(f"\n===== Experiment: {exp_name} | Seed: {seed} =====")
        set_seed(seed)
        hsi_use = hsi.copy()
        hsi_use, _ = apply_pca_if_needed(hsi_use, PCA_COMPONENTS, seed=seed)
        hsi_use = NormalizeHSI_MCMTN(hsi_use)
        rows, cols, labels = make_labeled_indices(gt)
        train_idx, val_idx, test_idx = stratified_split_indices(rows, cols, labels,train_ratio=exp_cfg['train_ratio'],val_ratio=exp_cfg['val_ratio'],seed=seed,)
        train_ds = HSIPatchDataset(hsi_use, gt, train_idx, patch_size=exp_cfg['patch_size'])
        val_ds = HSIPatchDataset(hsi_use, gt, val_idx, patch_size=exp_cfg['patch_size'])
        test_ds = HSIPatchDataset(hsi_use, gt, test_idx, patch_size=exp_cfg['patch_size'])
        train_loader = make_loader(train_ds, BATCH_SIZE, shuffle=True, drop_last=True)
        val_loader = make_loader(val_ds, BATCH_SIZE, shuffle=False, drop_last=False)
        test_loader = make_loader(test_ds, BATCH_SIZE, shuffle=False, drop_last=False)
        model = HSIWorldMambaNet(bands=HSI.shape[2],num_classes=len(class_names),patch_size=exp_cfg["patch_size"],
                                 model_dim=MODEL_DIM,depth=MAMBA_LAYERS,d_state=MAMBA_D_STATE,d_conv=MAMBA_D_CONV,
                                 expand=MAMBA_EXPAND,factor_dim=FACTOR_DIM,drop=DROPOUT,use_material=exp_cfg["use_material"],
                                 use_scene=exp_cfg["use_scene"],use_phys=exp_cfg["use_phys"],).to(DEVICE)
        model = maybe_compile_model(model)
        trainable_m = sum(p.numel() for p in model.parameters() if p.requires_grad) / 1e6
        print('Trainable params:', f'{trainable_m:.3f}', 'M')
        seed_dir = exp_dir / f'seed_{seed}'
        ensure_dir(seed_dir)
        model, history, test_metrics, y_true, y_pred, y_prob, coords, best_epoch, best_val_oa = train_model(model, train_loader, 
                                                                                                            val_loader, test_loader, class_names, exp_cfg, seed_dir)
        save_seed_run_artifacts(exp_dir=exp_dir,seed=seed,metrics=test_metrics,y_true=y_true,y_pred=y_pred,y_prob=y_prob,coords=coords,)
        seed_records.append({"experiment": exp_name,"seed": seed,"best_epoch": best_epoch,"best_val_oa": best_val_oa,
                             "oa": test_metrics["oa"],"aa": test_metrics["aa"],"kappa": test_metrics["kappa"],"seed_dir": str(seed_dir),})
        if best_val_oa > best_seed_val_oa:
            best_seed_val_oa = best_val_oa
            best_seed_info = {'seed': seed,'best_epoch': best_epoch,'best_val_oa': best_val_oa,'seed_dir': str(seed_dir),'exp_dir': str(exp_dir),}
            best_seed_model = model
            best_seed_test_pack = (hsi_use, gt)
        del model
        if DEVICE.type == 'cuda':
            torch.cuda.empty_cache()
        gc.collect()
    summary_df = pd.DataFrame(seed_records)
    summary_df.to_csv(exp_dir / "seed_wise_results.csv",index=False,)
    if SAVE_SCENE_MAPS and best_seed_info is not None and best_seed_model is not None:
        hsi_best, gt_best = best_seed_test_pack
        best_seed_dir = exp_dir / f"seed_{best_seed_info['seed']}"
        save_prediction_maps(best_seed_model,hsi_best,gt_best,best_seed_dir,patch_size=exp_cfg["patch_size"],class_names=class_names,)
    return summary_df
## Main execution
HSI, GT, NUM_CLASSES, CLASS_NAMES = LoadHSIData(HSID, DATA_ROOT)
print("HSI shape:", HSI.shape)
print("GT shape:", GT.shape)
print("Num classes:", NUM_CLASSES)
experiments = build_experiment_list()
print("Experiments to run:")
for exp in experiments:
    print(exp)
all_seed_records = []
for exp_cfg in experiments:
    summary_df = run_experiment(exp_cfg,SEED_LIST,HSI,GT,CLASS_NAMES,)
    summary_df.to_csv(RESULT_ROOT / f"{exp_cfg['name']}_seed_wise_results.csv",index=False,)
    all_seed_records.append(summary_df)
if all_seed_records:
    combined_seed_df = pd.concat(all_seed_records, ignore_index=True)
    combined_seed_df.to_csv(RESULT_ROOT / "all_experiments_seed_wise_results.csv",index=False,)
print("All experiments completed.")

Torch: 2.5.1+cu121
Device: cuda
Data root: /home/ahmad/repos/HSI
Result root: /home/ahmad/repos/graph_mamba/UH_Mamba_World_Results
AMP dtype: torch.bfloat16
Auto batch size: 32
Workers: 8
HSI shape: (349, 1905, 144)
GT shape: (349, 1905)
Num classes: 15
Experiments to run:
{'name': 'full_model', 'patch_size': 15, 'train_ratio': 0.3, 'val_ratio': 0.2, 'use_material': True, 'use_scene': True, 'use_phys': True, 'use_consistency': True, 'use_supcon': True}

===== Experiment: full_model | Seed: 44 =====
Trainable params: 0.357 M


W0730 14:54:53.958000 2680 site-packages/torch/_dynamo/convert_frame.py:1125] WON'T CONVERT forward /tmp/ipykernel_2680/4236382794.py line 398 
W0730 14:54:53.958000 2680 site-packages/torch/_dynamo/convert_frame.py:1125] due to: 
W0730 14:54:53.958000 2680 site-packages/torch/_dynamo/convert_frame.py:1125] Traceback (most recent call last):
W0730 14:54:53.958000 2680 site-packages/torch/_dynamo/convert_frame.py:1125]   File "/home/ahmad/miniconda3/envs/pt/lib/python3.10/site-packages/torch/_dynamo/output_graph.py", line 1446, in _call_user_compiler
W0730 14:54:53.958000 2680 site-packages/torch/_dynamo/convert_frame.py:1125]     compiled_fn = compiler_fn(gm, self.example_inputs())
W0730 14:54:53.958000 2680 site-packages/torch/_dynamo/convert_frame.py:1125]   File "/home/ahmad/miniconda3/envs/pt/lib/python3.10/site-packages/torch/_dynamo/repro/after_dynamo.py", line 129, in __call__
W0730 14:54:53.958000 2680 site-packages/torch/_dynamo/convert_frame.py:1125]     compiled_gm = compile

Epoch 001 | Loss 4.4919 | Val OA 0.6707 | AA 0.6871 | Kappa 0.6449
Epoch 002 | Loss 3.3839 | Val OA 0.7894 | AA 0.8042 | Kappa 0.7726
Epoch 003 | Loss 2.9602 | Val OA 0.8187 | AA 0.8360 | Kappa 0.8043
Epoch 004 | Loss 2.6725 | Val OA 0.8796 | AA 0.8874 | Kappa 0.8699
Epoch 005 | Loss 2.4750 | Val OA 0.8633 | AA 0.8803 | Kappa 0.8523
Epoch 006 | Loss 2.3214 | Val OA 0.8955 | AA 0.9074 | Kappa 0.8871
Epoch 007 | Loss 2.2101 | Val OA 0.9112 | AA 0.9207 | Kappa 0.9040
Epoch 008 | Loss 2.1305 | Val OA 0.9128 | AA 0.9225 | Kappa 0.9058
Epoch 009 | Loss 2.0696 | Val OA 0.9438 | AA 0.9461 | Kappa 0.9392
Epoch 010 | Loss 2.0303 | Val OA 0.9544 | AA 0.9565 | Kappa 0.9507
Epoch 011 | Loss 1.9536 | Val OA 0.9431 | AA 0.9478 | Kappa 0.9385
Epoch 012 | Loss 1.9111 | Val OA 0.9578 | AA 0.9602 | Kappa 0.9543
Epoch 013 | Loss 1.8663 | Val OA 0.9657 | AA 0.9670 | Kappa 0.9630
Epoch 014 | Loss 1.8417 | Val OA 0.9727 | AA 0.9725 | Kappa 0.9705
Epoch 015 | Loss 1.8118 | Val OA 0.9774 | AA 0.9762 | Kappa 0.

/tmp/ipykernel_2680/4236382794.py:775: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  ckpt = torch.load(best_path, map_location=DEVICE)


Best epoch: 65
Test OA: 0.9962741184298071
Test AA: 0.9954019458301975
Test Kappa: 0.995971990074273
